<a href="https://colab.research.google.com/github/giabq/plnEp2/blob/main/BiLSTM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Clonar o repositório e acessar os dados

In [8]:
# Clonar repositório
!git clone https://github.com/giabq/plnEp2.git

# Acessar diretório
%cd plnEp2

Cloning into 'plnEp2'...
remote: Enumerating objects: 68, done.
remote: Counting objects: 100% (68/68), done.
remote: Compressing objects: 100% (61/61), done.
remote: Total 68 (delta 25), reused 22 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (68/68), 13.88 MiB | 9.20 MiB/s, done.
Resolving deltas: 100% (25/25), done.
Updating files: 100% (18/18), done.
/content/plnEp2/plnEp2


#

# Instalação das bibliotecas

In [9]:
!pip install -q gensim
!pip install -q tqdm

In [10]:
# Manipulação de dados
import numpy as np
import pandas as pd
import re

# FastText
from gensim.models import FastText

# TensorFlow / Keras
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Input,
    Embedding,
    Bidirectional,
    LSTM,
    Dense,
    Dropout
)

# Scikit-learn
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

# Reprodutibilidade
SEED = 42

np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)
print("GPU disponível:", tf.config.list_physical_devices("GPU"))


TensorFlow: 2.21.0
GPU disponível: []


#Carregamento dos dados limpos

In [11]:
# Carregar dados limpos
train = pd.read_csv("data/train_limpo.csv")

# Remover registros sem texto ou classificação
train = train.dropna(subset=["resp_text", "clarity"])

# Padronizar rótulos
train["clarity"] = train["clarity"].astype(str).str.lower()

# Separar textos e rótulos
X_train = train["resp_text"].astype(str).values
y_train = train["clarity"].values

print("Quantidade de exemplos:", len(train))
print("\nDistribuição das classes:")
print(train["clarity"].value_counts())

print("\nExemplo de texto:")
print(X_train[0])

Quantidade de exemplos: 20092

Distribuição das classes:
clarity
c5      6892
c234    6853
c1      6347
Name: count, dtype: int64

Exemplo de texto:
prezado a senhor a esclarecemos que o serviço de informação ao cidadão sic não trata de sugestão elogio solicitação reclamação e denúncia relacionados a objetos postados no brasil ou no exterior cumpre nos esclarecer que a comissão mista de reavaliação de informações cmri por meio da súmula nº 01 2015 definiu caso exista canal ou procedimento específico efetivo para obtenção da informação solicitada o órgão ou a entidade deve orientar o interessado a buscar a informação por intermédio desse canal ou procedimento indicando os prazos e as condições para sua utilização sendo o pedido considerado atendido assim especificamente sobre o seu pedido de informação ressaltamos que foi identificado manifestação aberta na central de atendimento dos correios cac e nesse sentido ressaltamos que as informações sobre a sua remessa postal serão prestadas a

#Implementação do classificador BiLSTM + FastText

In [15]:
class BiLSTMFClassifier(BaseEstimator, ClassifierMixin):

    def __init__(
        self,
        lstm_units=32,
        dropout=0.3,
        learning_rate=0.001,
        vector_size=100,
        max_vocab_size=20000,
        max_sequence_length=300,
        fasttext_epochs=10,
        epochs=12,
        batch_size=32,
        random_state=42
    ):
        self.lstm_units = lstm_units
        self.dropout = dropout
        self.learning_rate = learning_rate
        self.vector_size = vector_size
        self.max_vocab_size = max_vocab_size
        self.max_sequence_length = max_sequence_length
        self.fasttext_epochs = fasttext_epochs
        self.epochs = epochs
        self.batch_size = batch_size
        self.random_state = random_state

    # Tokenização
    def _tokenize(self, text):
        return re.findall(
            r"\b\w+\b",
            text.lower(),
            flags=re.UNICODE
        )

    # Construção da rede neural
    def _build_model(self):

        tf.keras.backend.clear_session()

        model = Sequential([
            Input(shape=(self.max_sequence_length,)),

            Embedding(
                input_dim=self.vocab_size,
                output_dim=self.vector_size,
                weights=[self.embedding_matrix],
                mask_zero=True,
                trainable=True
            ),

            Bidirectional(
                LSTM(
                    self.lstm_units,
                    dropout=self.dropout
                )
            ),

            Dropout(self.dropout),

            Dense(32, activation="relu"),

            Dropout(self.dropout),

            Dense(
                len(self.classes_),
                activation="softmax"
            )
        ])

        model.compile(
            optimizer=tf.keras.optimizers.Adam(
                learning_rate=self.learning_rate
            ),
            loss="sparse_categorical_crossentropy",
            metrics=["accuracy"]
        )

        return model

    # Treinamento
    def fit(self, X, y):

        np.random.seed(self.random_state)
        tf.random.set_seed(self.random_state)

        X = np.asarray(X, dtype=str)
        y = np.asarray(y)

        print("\n" + "=" * 55, flush=True)
        print("INICIANDO NOVO TREINAMENTO", flush=True)
        print(f"Exemplos: {len(X)}", flush=True)
        print(f"Unidades LSTM: {self.lstm_units}", flush=True)
        print("=" * 55, flush=True)

        # Codificação das classes
        self.classes_ = np.unique(y)

        class_to_index = {
            label: index
            for index, label in enumerate(self.classes_)
        }

        y_encoded = np.array([
            class_to_index[label] for label in y
        ])

        # Tokenização
        print("\n[1/4] Preparando tokenizador...", flush=True)

        self.tokenizer = tf.keras.preprocessing.text.Tokenizer(
            num_words=self.max_vocab_size,
            oov_token="<OOV>"
        )

        self.tokenizer.fit_on_texts(X)

        sequences = self.tokenizer.texts_to_sequences(X)

        X_pad = tf.keras.preprocessing.sequence.pad_sequences(
            sequences,
            maxlen=self.max_sequence_length,
            padding="post",
            truncating="post"
        )

        self.vocab_size = min(
            self.max_vocab_size,
            len(self.tokenizer.word_index) + 1
        )

        print(
            f"Vocabulário: {self.vocab_size} palavras",
            flush=True
        )

        # Treinamento FastText
        print("\n[2/4] Treinando FastText...", flush=True)

        corpus = [
            self._tokenize(text)
            for text in X
        ]

        self.fasttext_model = FastText(
            sentences=corpus,
            vector_size=self.vector_size,
            window=5,
            min_count=1,
            workers=2,
            sg=1,
            min_n=3,
            max_n=6,
            epochs=self.fasttext_epochs,
            seed=self.random_state
        )

        print("FastText concluído!", flush=True)

        # Matriz de embeddings
        print(
            "\n[3/4] Construindo matriz de embeddings...",
            flush=True
        )

        self.embedding_matrix = np.zeros(
            (self.vocab_size, self.vector_size)
        )

        for word, index in self.tokenizer.word_index.items():

            if index >= self.vocab_size:
                continue

            if word == "<OOV>":
                continue

            self.embedding_matrix[index] = (
                self.fasttext_model.wv[word]
            )

        print(
            "Matriz de embeddings:",
            self.embedding_matrix.shape,
            flush=True
        )

        # Construção e treinamento BiLSTM
        print("\n[4/4] Treinando rede BiLSTM...", flush=True)

        self.model = self._build_model()

        self.model.fit(
            X_pad,
            y_encoded,
            epochs=self.epochs,
            batch_size=self.batch_size,
            verbose=2,
            shuffle=True
        )

        print("\nTREINAMENTO CONCLUÍDO!", flush=True)

        return self

    # Probabilidades de classificação
    def predict_proba(self, X):

        X = np.asarray(X, dtype=str)

        sequences = self.tokenizer.texts_to_sequences(X)

        X_pad = tf.keras.preprocessing.sequence.pad_sequences(
            sequences,
            maxlen=self.max_sequence_length,
            padding="post",
            truncating="post"
        )

        return self.model.predict(
            X_pad,
            verbose=0
        )

    # Predição das classes
    def predict(self, X):

        probabilities = self.predict_proba(X)

        predicted_indices = np.argmax(
            probabilities,
            axis=1
        )

        return self.classes_[predicted_indices]

#Definição do modelo e dos hiperparâmetros

In [16]:
# Modelo-base
bilstm = BiLSTMFClassifier(
    random_state=SEED
)

# Grade de hiperparâmetros
param_grid = {
    "lstm_units": [32, 64],
    "dropout": [0.3]
}

# Validação cruzada estratificada com 3 folds
cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=SEED
)

print("Combinações:", 2)
print("Folds:", 3)
print("Total de treinamentos:", 2 * 3)

Combinações: 2
Folds: 3
Total de treinamentos: 6



#Execução do Grid Search com validação cruzada

In [17]:
import time

# Configuração do Grid Search
grid_search = GridSearchCV(
    estimator=bilstm,
    param_grid=param_grid,
    scoring={
        "accuracy": "accuracy",
        "f1_macro": "f1_macro"
    },
    refit="accuracy",
    cv=cv,
    n_jobs=1,
    verbose=2,
    return_train_score=True,
    error_score="raise"
)

# Quantidade de treinamentos da validação cruzada
total_fits = (
    len(param_grid["lstm_units"]) *
    cv.get_n_splits()
)

print("=" * 60)
print("INICIANDO GRID SEARCH - BiLSTM + FastText")
print("=" * 60)

print(f"Combinações de hiperparâmetros: {len(param_grid['lstm_units'])}")
print(f"Folds: {cv.get_n_splits()}")
print(f"Treinamentos da validação cruzada: {total_fits}")
print("Critério de seleção: Accuracy")
print("=" * 60)

inicio = time.time()

# Executar Grid Search
grid_search.fit(X_train, y_train)

# Calcular tempo total
tempo_total = (time.time() - inicio) / 60

# Resultados
print("\n" + "=" * 60)
print("GRID SEARCH CONCLUÍDO!")
print("=" * 60)

print(f"Tempo total: {tempo_total:.2f} minutos")

print("\nMelhores hiperparâmetros:")
print(grid_search.best_params_)

print(
    f"\nMelhor Accuracy média: "
    f"{grid_search.best_score_:.4f}"
)

print("\nMacro-F1 médio da melhor configuração:")

best_index = grid_search.best_index_

print(
    f"{grid_search.cv_results_['mean_test_f1_macro'][best_index]:.4f}"
)

INICIANDO GRID SEARCH - BiLSTM + FastText
Combinações de hiperparâmetros: 2
Folds: 3
Treinamentos da validação cruzada: 6
Critério de seleção: Accuracy
Fitting 3 folds for each of 2 candidates, totalling 6 fits

INICIANDO NOVO TREINAMENTO
Exemplos: 13394
Unidades LSTM: 32

[1/4] Preparando tokenizador...
Vocabulário: 20000 palavras

[2/4] Treinando FastText...
FastText concluído!

[3/4] Construindo matriz de embeddings...
Matriz de embeddings: (20000, 100)

[4/4] Treinando rede BiLSTM...
Epoch 1/12
419/419 - 149s - 356ms/step - accuracy: 0.3581 - loss: 1.0994
Epoch 2/12
419/419 - 205s - 490ms/step - accuracy: 0.3908 - loss: 1.0822
Epoch 3/12
419/419 - 143s - 341ms/step - accuracy: 0.4182 - loss: 1.0669
Epoch 4/12
419/419 - 204s - 487ms/step - accuracy: 0.4370 - loss: 1.0449
Epoch 5/12
419/419 - 199s - 476ms/step - accuracy: 0.4756 - loss: 1.0143
Epoch 6/12
419/419 - 143s - 341ms/step - accuracy: 0.5061 - loss: 0.9696
Epoch 7/12
419/419 - 202s - 483ms/step - accuracy: 0.5507 - loss: 0.9

#Resultados do Grid Search

In [18]:
results = pd.DataFrame(grid_search.cv_results_)

display(
    results[
        [
            "params",
            "mean_test_accuracy",
            "std_test_accuracy",
            "mean_test_f1_macro",
            "std_test_f1_macro"
        ]
    ].sort_values(
        by="mean_test_accuracy",
        ascending=False
    )
)

,params,mean_test_accuracy,std_test_accuracy,mean_test_f1_macro,std_test_f1_macro
1,"{'dropout': 0.3, 'lstm_units': 64}",0.418176,0.003450,0.418641,0.003765
0,"{'dropout': 0.3, 'lstm_units': 32}",0.411855,0.006216,0.411230,0.008150


#Conferência dos resultados da validação cruzada

In [19]:
print("RESULTADOS DA VALIDAÇÃO CRUZADA")
print("=" * 45)

best_index = grid_search.best_index_

for i in range(cv.get_n_splits()):

    print(f"\nFold {i + 1}")

    print(
        "Accuracy:",
        f"{grid_search.cv_results_[f'split{i}_test_accuracy'][best_index]:.4f}"
    )

    print(
        "Macro-F1:",
        f"{grid_search.cv_results_[f'split{i}_test_f1_macro'][best_index]:.4f}"
    )

print("\nMÉDIAS")
print("=" * 45)

print(
    "Accuracy:",
    f"{grid_search.cv_results_['mean_test_accuracy'][best_index]:.4f}"
)

print(
    "Macro-F1:",
    f"{grid_search.cv_results_['mean_test_f1_macro'][best_index]:.4f}"
)

RESULTADOS DA VALIDAÇÃO CRUZADA

Fold 1
Accuracy: 0.4230
Macro-F1: 0.4240

Fold 2
Accuracy: 0.4150
Macro-F1: 0.4160

Fold 3
Accuracy: 0.4166
Macro-F1: 0.4160

MÉDIAS
Accuracy: 0.4182
Macro-F1: 0.4186


#Carregamento do Teste 1 para avaliação posterior

In [20]:
# Carregar Teste 1 rotulado
test1_rotulado = pd.read_excel(
    "data/test1-rotulado-para-teste-estatistico.xlsx"
)

# Carregar textos limpos do Teste 1
test1_limpo = pd.read_csv(
    "data/test1_limpo.csv"
)

print("Teste 1 rotulado:")
print(test1_rotulado.shape)
print(test1_rotulado.columns.tolist())

print("\nTeste 1 limpo:")
print(test1_limpo.shape)
print(test1_limpo.columns.tolist())

print("\nPrimeiras linhas do arquivo rotulado:")
display(test1_rotulado.head())

print("\nPrimeiras linhas do arquivo limpo:")
display(test1_limpo.head())

Teste 1 rotulado:
(900, 2)
['resp_text', 'clarity']

Teste 1 limpo:
(900, 2)
['resp_text', 'clarity']

Primeiras linhas do arquivo rotulado:


,resp_text,clarity
0,"Prezada Hellen, Dentre as pesquisas domici...",c5
1,"Senhor Francisco, O Serviço de Informações a...",c1
2,"Prezada Srª. Raylane, Em resposta a sua solic...",c5
3,"Prezada Senhora, De acordo com o art.13 do D...",c1
4,"Prezado(a) Senhor(a), Em atendimento ao ped...",c234



Primeiras linhas do arquivo limpo:


,resp_text,clarity
0,prezada hellen dentre as pesquisas domiciliare...,c5
1,senhor francisco o serviço de informações ao c...,c1
2,prezada srª raylane em resposta a sua solicita...,c5
3,prezada senhora de acordo com o art 13 do decr...,c1
4,prezado a senhor a em atendimento ao pedido de...,c234


#Avaliação do BiLSTM + FastText no Test 1

In [21]:
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)
import pandas as pd

# Textos e rótulos do Test 1
X_test1 = test1_limpo["resp_text"].fillna("").astype(str).values
y_test1 = test1_rotulado["clarity"].astype(str).str.lower().values

# Conferir alinhamento dos dados
assert len(X_test1) == len(y_test1), "Quantidade de exemplos incompatível!"

if "clarity" in test1_limpo.columns:
    y_test1_limpo = test1_limpo["clarity"].astype(str).str.lower().values
    assert (y_test1 == y_test1_limpo).all(), "Os rótulos não estão alinhados!"

# Recuperar o melhor modelo treinado pelo Grid Search
best_model = grid_search.best_estimator_

# Realizar previsões
y_pred_test1 = best_model.predict(X_test1)

# Calcular métricas
accuracy = accuracy_score(y_test1, y_pred_test1)
macro_f1 = f1_score(y_test1, y_pred_test1, average="macro")

print("=" * 55)
print("AVALIAÇÃO - BiLSTM + FastText (TEST 1)")
print("=" * 55)

print(f"Accuracy: {accuracy:.4f}")
print(f"Macro-F1: {macro_f1:.4f}")

print("\nRelatório de classificação:")
print(
    classification_report(
        y_test1,
        y_pred_test1,
        labels=["c1", "c234", "c5"],
        zero_division=0
    )
)

print("\nMatriz de confusão:")
cm = confusion_matrix(
    y_test1,
    y_pred_test1,
    labels=["c1", "c234", "c5"]
)

display(
    pd.DataFrame(
        cm,
        index=["Real c1", "Real c234", "Real c5"],
        columns=["Predito c1", "Predito c234", "Predito c5"]
    )
)

AVALIAÇÃO - BiLSTM + FastText (TEST 1)
Accuracy: 0.4467
Macro-F1: 0.4479

Relatório de classificação:
              precision    recall  f1-score   support

          c1       0.48      0.46      0.47       300
        c234       0.37      0.50      0.43       300
          c5       0.54      0.38      0.44       300

    accuracy                           0.45       900
   macro avg       0.47      0.45      0.45       900
weighted avg       0.47      0.45      0.45       900


Matriz de confusão:


,Predito c1,Predito c234,Predito c5
Real c1,139,123,38
Real c234,92,150,58
Real c5,56,131,113
